In [ ]:
import numpy as np
import pandas as pd
import pickle
import anndata
import os, sys
import pyBigWig
from scipy.stats import mannwhitneyu
from plotnine import *

sys.path.append('/code/decima/src/decima')
sys.path.append('/hpc/mydata/mathias.voges/Projects/research/seq2fun/step/daniodecima-main/src/decima/')

#from resources import load_gtf

%matplotlib inline

## Paths

In [ ]:

# Paste the save_dirs and model_labels extracted from your PAIRS array here:
save_dirs = [
"/hpc/scratch/group.data.science/mathias.voges/zebrahub-decima/experiments/decima_experiments_20250618_111138/pretrained_decima-human_rep0_lr3e-05_seed42/version_0",
"/hpc/scratch/group.data.science/mathias.voges/zebrahub-decima/experiments/decima_experiments_20250618_111138/pretrained_decima-human_rep1_lr3e-05_seed42/version_0",
"/hpc/scratch/group.data.science/mathias.voges/zebrahub-decima/experiments/decima_experiments_20250618_111138/pretrained_decima-human_rep2_lr3e-05_seed42/version_0",
"/hpc/scratch/group.data.science/mathias.voges/zebrahub-decima/experiments/decima_experiments_20250618_111138/pretrained_decima-human_rep3_lr3e-05_seed42/version_0",
"/hpc/scratch/group.data.science/mathias.voges/zebrahub-decima/experiments/decima_experiments_20250617_225114/pretrained_wandb-human_rep0_lr3e-05_seed42/version_0",
"/hpc/scratch/group.data.science/mathias.voges/zebrahub-decima/experiments/decima_experiments_20250617_225114/pretrained_wandb-human_rep1_lr3e-05_seed42/version_0",
"/hpc/scratch/group.data.science/mathias.voges/zebrahub-decima/experiments/decima_experiments_20250617_225114/pretrained_wandb-human_rep2_lr3e-05_seed42/version_0",
"/hpc/scratch/group.data.science/mathias.voges/zebrahub-decima/experiments/decima_experiments_20250617_225114/pretrained_wandb-human_rep3_lr3e-05_seed42/version_0",
"/hpc/scratch/group.data.science/mathias.voges/zebrahub-decima/experiments/decima_experiments_20250617_225114/pretrained_wandb-mouse_rep0_lr3e-05_seed42/version_0",
"/hpc/scratch/group.data.science/mathias.voges/zebrahub-decima/experiments/decima_experiments_20250617_225114/pretrained_wandb-mouse_rep1_lr3e-05_seed42/version_0",
"/hpc/scratch/group.data.science/mathias.voges/zebrahub-decima/experiments/decima_experiments_20250617_225114/pretrained_wandb-mouse_rep2_lr3e-05_seed42/version_0",
"/hpc/scratch/group.data.science/mathias.voges/zebrahub-decima/experiments/decima_experiments_20250617_225114/pretrained_wandb-mouse_rep3_lr3e-05_seed42/version_0",
"/hpc/scratch/group.data.science/mathias.voges/zebrahub-decima/experiments/decima_experiments_20250617_225114/random_lr3e-06_seed42/version_0",
"/hpc/scratch/group.data.science/mathias.voges/zebrahub-decima/experiments/decima_experiments_20250617_225114/random_lr3e-06_seed43/version_0",
"/hpc/scratch/group.data.science/mathias.voges/zebrahub-decima/experiments/decima_experiments_20250617_225114/random_lr3e-06_seed44/version_0",
"/hpc/scratch/group.data.science/mathias.voges/zebrahub-decima/experiments/decima_experiments_20250617_225114/random_lr3e-06_seed45/version_0",

]

model_labels = [
    "Human_Decima_0",
    "Human_Decima_1",
    "Human_Decima_2",
    "Human_Decima_3",
    "Human_Borzoi_0",
    "Human_Borzoi_1",
    "Human_Borzoi_2",
    "Human_Borzoi_3",
    "Mouse_Borzoi_0",
    "Mouse_Borzoi_1",
    "Mouse_Borzoi_2",
    "Mouse_Borzoi_3",
    "Random_0",
    "Random_1",
    "Random_2",
    "Random_3"
]

# Load data_out and attribution files
anndatas = {}
attr_files = {}
cres_files_16hpf = {}
cres_files_all = {}

for save_dir, label in zip(save_dirs, model_labels):
    # Find the data_out file
    data_out_files = [f for f in os.listdir(save_dir) if f.startswith("data_out_decima") and f.endswith(".h5ad")]
    if data_out_files:
        ad = anndata.read_h5ad(os.path.join(save_dir, data_out_files[0]))
        anndatas[label] = ad
    else:
        print(f"No data_out file found in {save_dir}")
    # Find the attribution file
    attr_file_candidates = [f for f in os.listdir(save_dir) if f.endswith("-attr-th05-16hpf.h5")]
    if attr_file_candidates:
        attr_files[label] = os.path.join(save_dir, attr_file_candidates[0])
    else:
        print(f"No attribution file found in {save_dir}")
    
    # Find the cres_16hpf file
    cres_file_candidates = [f for f in os.listdir(save_dir) if f.endswith("cres_16hpf.pkl")]
    if cres_file_candidates:
        cres_files_16hpf[label] = os.path.join(save_dir, cres_file_candidates[0])
    else:
        print(f"No cres_16hpf file found in {save_dir}")

    # Find the cres_all file
    cres_file_candidates = [f for f in os.listdir(save_dir) if f.endswith("cres_all.pkl")]
    if cres_file_candidates:
        cres_files_all[label] = os.path.join(save_dir, cres_file_candidates[0])
    else:
        print(f"No cres_all file found in {save_dir}")

# Load the raw ATAC bigWig file
cwd = '/hpc/projects/data.science/yangjoon.kim/zebrahub_multiome/data/processed_data/TDR118reseq/outs'
bigwig_path = os.path.join(cwd, "atac_cut_sites.bigwig")
bw = pyBigWig.open(bigwig_path)

print("Loaded AnnData objects:", list(anndatas.keys()))
print("Loaded attribution files:", list(attr_files.keys()))
print("Loaded ATAC bigWig file:", bigwig_path)
print("Loaded cres_16hpf files:", list(cres_files_16hpf.keys()))
print("Loaded cres_all files:", list(cres_files_all.keys()))


In [ ]:
genes_combined_list = []
for label, file_path in cres_files_16hpf.items():
    with open(file_path, 'rb') as f:
        content = pickle.load(f)
        df = content['genes']
    df['Source'] = label
    genes_combined_list.append(df)

genes_combined_16hpf = pd.concat(genes_combined_list, ignore_index=True)

In [ ]:
genes_combined_list = []
for label, file_path in cres_files_all.items():
    with open(file_path, 'rb') as f:
        content = pickle.load(f)
        df = content['genes']
    df['Source'] = label
    genes_combined_list.append(df)

genes_combined_all = pd.concat(genes_combined_list, ignore_index=True)

In [ ]:
# 1. Load the list of highly variable genes
with open("/hpc/mydata/mathias.voges/Projects/research/seq2fun/step/daniodecima-applications-main/notebooks/2_dataset/highly_variable_genes.txt", "r") as f:
    hvg_list = [line.strip() for line in f]

# 2. Subset your DataFrame using the 'index' column
genes_combined_16hpf = genes_combined_16hpf[genes_combined_16hpf['index'].isin(hvg_list)]
genes_combined_all = genes_combined_all[genes_combined_all['index'].isin(hvg_list)]

# 3. (Optional) Check the result
print(genes_combined_16hpf.shape)
print(genes_combined_16hpf.head())

In [ ]:
genes_combined_all

### Overlap ATAC-peaks with attribution scores peaks. Figure 8 in the DanioDecima manuscript.

In [ ]:
def prepare_atac_vs_nonatac_df(genes_combined, region_pairs):
    differences = []
    for model in genes_combined['Source'].unique():
        model_data = genes_combined[genes_combined['Source'] == model]
        for atac_col, non_atac_col in region_pairs:
            atac_vals = model_data[atac_col].dropna()
            non_atac_vals = model_data[non_atac_col].dropna()
            if len(atac_vals) > 0 and len(non_atac_vals) > 0:
                print(len(atac_vals), len(non_atac_vals))
                print(atac_col, non_atac_col)
                median_atac = atac_vals.median()
                median_non_atac = non_atac_vals.median()
                fold_change = np.log2(median_atac / median_non_atac)
                region_name = atac_col.split(' ')[0] if '(' not in atac_col else atac_col.split(' (')[0]
                
                # Extract model number
                model_num = model.split('_')[-1] if '_' in model else '0'
                
                # Determine model type based on model name
                if model.startswith('Random'):
                    model_type = 'Random'
                    model_label = f"Random {model_num}"
                elif model.startswith('Human_Borzoi'):
                    model_type = 'Human-Borzoi'
                    model_label = f"Human-Borzoi {model_num}"
                elif model.startswith('Human_Decima'):
                    model_type = 'Human-Decima'
                    model_label = f"Human-Decima {model_num}"
                elif model.startswith('Mouse_Borzoi'):
                    model_type = 'Mouse-Borzoi'
                    model_label = f"Mouse-Borzoi {model_num}"
                else:
                    # Fallback for any unexpected naming
                    model_type = 'Unknown'
                    model_label = model
                
                differences.append({
                    'Model': model_label,
                    'Region': region_name,
                    'Fold_Change': fold_change,
                    'Type': model_type,
                    'ATAC_median': median_atac,
                    'non_ATAC_median': median_non_atac,
                    'Replicate': model_num
                })
    return pd.DataFrame(differences)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu
import matplotlib.patches as mpatches
import numpy as np

# Define the regions to compare
region_pairs = [
    ('Promoter CREs', 'Promoter non-CREs'),
    ('Exons CREs', 'Exons non-CREs'),
    ('Intronic CREs', 'Intronic non-CREs'),
    ('1k (CREs)', '1k (non-CREs)'),
    ('1k-10k (CREs)', '1k-10k (non-CREs)'),
    ('10k-100k (CREs)', '10k-100k (non-CREs)'),
    ('>=100k (CREs)', '>=100k (non-CREs)')
]

diff_df = prepare_atac_vs_nonatac_df(genes_combined_16hpf, region_pairs)

# Create figure with better proportions
plt.figure(figsize=(18, 8))

# Define colors for all four model types with better contrast
palette = {
    'Random': '#666666',
    'Human-Borzoi': '#e74c3c',
    'Human-Decima': '#8b0000',
    'Mouse-Borzoi': '#3498db'
}

# Create the plot
ax = plt.gca()

# Boxplot with improved styling
box_plot = sns.boxplot(
    data=diff_df,
    x='Region',
    y='Fold_Change',
    hue='Type',
    palette=palette,
    showfliers=False,
    width=0.7,
    linewidth=1.5,
    ax=ax
)

# Stripplot with better positioning
strip_plot = sns.stripplot(
    data=diff_df,
    x='Region',
    y='Fold_Change',
    hue='Type',
    palette=palette,
    dodge=True,
    size=10,
    alpha=0.8,
    edgecolor='white',
    linewidth=0.5,
    ax=ax
)

# Remove the automatic legends
if ax.legend_:
    ax.legend_.remove()

# Dynamically count models for each type
model_counts = diff_df.groupby('Type')['Model'].nunique()
type_order = ['Random', 'Human-Borzoi', 'Human-Decima', 'Mouse-Borzoi']
types_present = [t for t in type_order if t in diff_df['Type'].unique()]

# Create custom legend
legend_patches = []
for model_type in types_present:
    count = model_counts[model_type]
    color = palette[model_type]
    patch = mpatches.Patch(color=color, label=f'{model_type} (n={count})')
    legend_patches.append(patch)

# Position legend outside plot area
plt.legend(handles=legend_patches, 
          title='Model Type', 
          fontsize=12, 
          title_fontsize=14, 
          bbox_to_anchor=(1.02, 1), 
          loc='upper left',
          frameon=True,
          fancybox=True,
          shadow=True)

# Improve plot styling
plt.title('ATAC vs non-ATAC Attribution Fold Change by Region', 
          fontsize=18, fontweight='bold', pad=20)
plt.xlabel('Genomic Region', fontsize=14, fontweight='bold')
plt.ylabel('log₂(ATAC/non-ATAC)', fontsize=14, fontweight='bold')

# Fix x-axis labels
region_labels = [
    'Promoter',
    'Exons', 
    'Intronic',
    '1k',
    '1k-10k',
    '10k-100k',
    '>=100k'
]

plt.xticks(range(len(region_labels)), region_labels, 
           rotation=45, ha='right', fontsize=12)
plt.yticks(fontsize=12)

# Add reference line at y=0
plt.axhline(y=0, color='black', linestyle='--', alpha=0.5, linewidth=1)

# Add subtle grid
plt.grid(True, alpha=0.2, linestyle='-', linewidth=0.5)

# Set y-axis limits with some padding
y_min, y_max = diff_df['Fold_Change'].min(), diff_df['Fold_Change'].max()
y_range = y_max - y_min
plt.ylim(y_min - 0.1 * y_range, y_max + 0.1 * y_range)

# Improve layout
plt.tight_layout()

# Add subtle background color
ax.set_facecolor('#fafafa')

# Save with high quality
plt.savefig('atac_vs_non_atac_fold_change_by_type_all_models.png', 
           dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

# Print summary statistics with better formatting
print("="*50)
print("SUMMARY STATISTICS")
print("="*50)
print("\nModel type counts:")
for model_type, count in model_counts.items():
    print(f"  {model_type}: {count}")

print("\nMean fold changes by model type:")
mean_changes = diff_df.groupby('Type')['Fold_Change'].mean()
for model_type, mean_val in mean_changes.items():
    print(f"  {model_type}: {mean_val:.3f}")

print("\nMean fold changes by region:")
region_means = diff_df.groupby('Region')['Fold_Change'].mean()
for region, mean_val in region_means.items():
    print(f"  {region}: {mean_val:.3f}")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu
import pandas as pd

# 1. Select regions: Promoter, Exons, Introns, Exon/Intron junctions, and all columns ending with (all)
base_regions = ['Promoter', 'Exons', 'Introns', 'Exon/Intron junctions']
all_cols = genes_combined_all.columns.tolist()
all_region_cols = base_regions + [col for col in all_cols if col.endswith('(all)')]

# 2. Melt the DataFrame for plotting
df_all_melted = genes_combined_all.melt(
    id_vars=['Source'],
    value_vars=all_region_cols,
    var_name='Region',
    value_name='Attribution'
)

# 3. Use actual model labels from your data
model_labels = genes_combined_all['Source'].unique().tolist()
model_palette = {}
for label in model_labels:
    if label.startswith('Random'):
        model_palette[label] = 'grey'
    else:
        model_palette[label] = 'red'
hue_order = model_labels

# 4. Add Model column for plotting
df_all_melted['Model'] = df_all_melted['Source']

# 5. Calculate percentile ranks within each model
def calculate_percentile_ranks(df):
    return df.groupby('Model').apply(
        lambda x: x.assign(
            Percentile_Rank=x['Attribution'].rank(pct=True) * 100
        )
    ).reset_index(drop=True)

df_all_melted_ranked = calculate_percentile_ranks(df_all_melted)

# 6. Set region order (optional: by mean attribution)
region_order = df_all_melted_ranked.groupby('Region')['Attribution'].mean().sort_values(ascending=False).index.tolist()
df_all_melted_ranked['Region'] = pd.Categorical(df_all_melted_ranked['Region'], categories=region_order, ordered=True)

# 7. Plot
#plt.figure(figsize=(max(12, len(region_order)*1.2), 7))
plt.figure(figsize=(20, 7))
ax = sns.boxplot(
    data=df_all_melted_ranked,
    x='Region',
    y='Percentile_Rank',
    hue='Model',
    palette=model_palette,
    showfliers=False,
    hue_order=hue_order
)

plt.ylim(-10, 110)
plt.title('Percentile Rank of Attribution Scores by Genomic Region and Model', fontsize=18)
plt.xlabel('Genomic Region', fontsize=16)
plt.ylabel('Percentile Rank', fontsize=16)
plt.xticks(rotation=45, ha='right', fontsize=12)
plt.yticks(fontsize=14)

# Place legend outside the plot
handles, labels = ax.get_legend_handles_labels()
# source_counts = df_all_melted_ranked['Model'].value_counts().to_dict()
# new_labels = [f"{label} (n={source_counts.get(label, 0)})" for label in labels]
n_genes = genes_combined_all['Source'].value_counts().to_dict()
new_labels = [f"{label} (n={n_genes.get(label, 0)})" for label in labels]

plt.legend(
    handles=handles,
    labels=new_labels,
    title='Model',
    fontsize=14,
    title_fontsize=12,
    loc='upper left',
    bbox_to_anchor=(1.01, 1)
)
plt.tight_layout(rect=[0, 0, 0.85, 1])

# Statistical significance: Mann-Whitney U test for Human vs Random for each region
# Statistical significance: Mann-Whitney U test for Human vs Random for each region

regions = df_all_melted_ranked['Region'].cat.categories
for i, region in enumerate(regions):
    human_vals = df_all_melted_ranked[
        (df_all_melted_ranked['Region'] == region) & 
        (df_all_melted_ranked['Model'].str.startswith('Human'))
    ]['Percentile_Rank'].dropna()
    random_vals = df_all_melted_ranked[
        (df_all_melted_ranked['Region'] == region) & 
        (df_all_melted_ranked['Model'].str.startswith('Random'))
    ]['Percentile_Rank'].dropna()
    # Only annotate if both groups have at least 2 values and are not all identical
    if (
        len(human_vals) >= 2 and len(random_vals) >= 2 and
        (human_vals.nunique() > 1 or random_vals.nunique() > 1)
    ):
        stat, pval = mannwhitneyu(human_vals, random_vals, alternative='two-sided')
        if not np.isnan(pval):
            y_max = max(human_vals.max(), random_vals.max())
            plt.text(
                i, y_max*1.05, f"p={pval:.2e}", ha='center', va='bottom', fontsize=10, color='black'
            )
    # Otherwise, skip annotation (no p=nan will be shown)

plt.savefig('all_gene_regions_comparison_models_percentile.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
import numpy as np

def calculate_entropy(attributions):
    """Calculate normalized entropy - 0 = perfectly focused, 1 = perfectly uniform"""
    # Normalize to probabilities
    probs = attributions / attributions.sum()
    # Calculate entropy
    entropy = -np.sum(probs * np.log(probs + 1e-10))  # Add small value to avoid log(0)
    # Normalize by max possible entropy (uniform distribution)
    max_entropy = np.log(len(probs))
    return entropy / max_entropy

def get_model_type(model_name):
    """Determine model type from model name"""
    if model_name.startswith('Random'):
        return 'Random'
    elif model_name.startswith('Human_Borzoi'):
        return 'Human-Borzoi'
    elif model_name.startswith('Human_Decima'):
        return 'Human-Decima'
    elif model_name.startswith('Mouse_Borzoi'):
        return 'Mouse-Borzoi'
    else:
        return 'Unknown'

# Calculate entropy for each model
model_entropy = []
for model in df_all_melted['Model'].unique():
    model_data = df_all_melted[df_all_melted['Model'] == model]
    region_means = model_data.groupby('Region')['Attribution'].mean()
    entropy = calculate_entropy(region_means.values)
    model_type = get_model_type(model)
    
    model_entropy.append({
        'Model': model, 
        'Attention_Entropy': entropy,
        'Model_Type': model_type
    })

entropy_df = pd.DataFrame(model_entropy)

# Define colors for all model types
palette = {
    'Random': 'grey',
    'Human-Borzoi': 'red',
    'Human-Decima': 'darkred',
    'Mouse-Borzoi': 'blue'
}

plt.figure(figsize=(12, 7))
sns.barplot(data=entropy_df, x='Model', y='Attention_Entropy', hue='Model_Type',
            palette=palette)
plt.title('Attention Entropy Across Model Types', fontsize=16)
plt.ylabel('Attention Entropy\n(0 = Highly Focused, 1 = Uniform)', fontsize=12)
plt.xlabel('Model', fontsize=12)
plt.ylim(0, 1)
plt.xticks(rotation=45, ha='right')
plt.axhline(y=0.5, color='black', linestyle='--', alpha=0.5, label='Moderate Focus (0.5)')

# Improve legend
handles, labels = plt.gca().get_legend_handles_labels()
# Remove the axhline from legend and add it manually if needed
legend_handles = handles[:-1] if 'Moderate Focus' in labels else handles
legend_labels = labels[:-1] if 'Moderate Focus' in labels else labels

plt.legend(legend_handles, legend_labels, title='Model Type', 
           bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.savefig('attention_entropy_by_model_type.png', dpi=300, bbox_inches='tight')
plt.show()

# Print summary statistics
print("Attention Entropy Summary:")
print(entropy_df.groupby('Model_Type')['Attention_Entropy'].agg(['mean', 'std']).round(3))

### Raw attribution score visualization by genomic region. Figure 7 in the DanioDecima manuscript.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu
import pandas as pd

# Use the same melted DataFrame as before
df_all_melted['Model'] = df_all_melted['Source']

# Set region order (optional: by mean attribution)
region_order = df_all_melted.groupby('Region')['Attribution'].mean().sort_values(ascending=False).index.tolist()
df_all_melted['Region'] = pd.Categorical(df_all_melted['Region'], categories=region_order, ordered=True)

# Create a simplified color mapping - one color per model group
model_labels = genes_combined_all['Source'].unique().tolist()
model_palette = {}

for label in model_labels:
    if 'Random' in label:
        model_palette[label] = '#666666'  # Grey for all Random models
    elif 'Human_Borzoi' in label or 'Human-Borzoi' in label:
        model_palette[label] = '#e74c3c'  # Red for all Human-Borzoi models
    elif 'Human_Decima' in label or 'Human-Decima' in label:
        model_palette[label] = '#8b0000'  # Dark red for all Human-Decima models
    elif 'Mouse' in label:
        model_palette[label] = '#3498db'  # Blue for all Mouse models
    else:
        model_palette[label] = '#95a5a6'  # Default grey

hue_order = model_labels

plt.figure(figsize=(20, 7))
ax = sns.boxplot(
    data=df_all_melted,
    x='Region',
    y='Attribution',
    hue='Model',
    palette=model_palette,
    showfliers=False,
    hue_order=hue_order
)

plt.title('Raw Attribution Scores by Genomic Region and Model', fontsize=18, fontweight='bold', pad=20)
plt.xlabel('Genomic Region', fontsize=16, fontweight='bold')
plt.ylabel('Raw Attribution Score', fontsize=16, fontweight='bold')
plt.xticks(rotation=45, ha='right', fontsize=12)
plt.yticks(fontsize=14)
plt.yscale('log')

# Get the current y-limits before adding p-values
current_ylim = ax.get_ylim()
data_max = df_all_melted['Attribution'].max()

# Set initial y-limits with some space for p-values
plt.ylim(bottom=1e-6, top=data_max * 10)  # More space at top for p-values

# Create custom legend with model group information
handles, labels = ax.get_legend_handles_labels()
n_genes = genes_combined_all['Source'].value_counts().to_dict()

# Group models by type for better legend organization
model_groups = {
    'Random': [],
    'Human-Borzoi': [],
    'Human-Decima': [],
    'Mouse-Borzoi': []
}

for i, label in enumerate(labels):
    count = n_genes.get(label, 0)
    if 'Random' in label:
        model_groups['Random'].append((handles[i], f"{label} (n={count})"))
    elif 'Human_Borzoi' in label or 'Human-Borzoi' in label:
        model_groups['Human-Borzoi'].append((handles[i], f"{label} (n={count})"))
    elif 'Human_Decima' in label or 'Human-Decima' in label:
        model_groups['Human-Decima'].append((handles[i], f"{label} (n={count})"))
    elif 'Mouse' in label:
        model_groups['Mouse-Borzoi'].append((handles[i], f"{label} (n={count})"))

# Flatten the grouped items back into handles and labels
final_handles = []
final_labels = []
for group_name in ['Random', 'Human-Borzoi', 'Human-Decima', 'Mouse-Borzoi']:
    if model_groups[group_name]:  # Only add if group has models
        for handle, label in model_groups[group_name]:
            final_handles.append(handle)
            final_labels.append(label)

plt.legend(
    handles=final_handles,
    labels=final_labels,
    title='Model Type',
    fontsize=11,
    title_fontsize=13,
    loc='upper left',
    bbox_to_anchor=(1.02, 1),
    frameon=True,
    fancybox=True,
    shadow=True
)

# Add subtle grid
plt.grid(True, alpha=0.2, linestyle='-', linewidth=0.5)

# Set background color
ax.set_facecolor('#fafafa')

# Statistical significance: Mann-Whitney U test for Human vs Random for each region
regions = df_all_melted['Region'].cat.categories
for i, region in enumerate(regions):
    human_vals = df_all_melted[
        (df_all_melted['Region'] == region) & 
        (df_all_melted['Model'].str.contains('Human'))
    ]['Attribution'].dropna()
    random_vals = df_all_melted[
        (df_all_melted['Region'] == region) & 
        (df_all_melted['Model'].str.contains('Random'))
    ]['Attribution'].dropna()
    
    if (
        len(human_vals) >= 2 and len(random_vals) >= 2 and
        (human_vals.nunique() > 1 or random_vals.nunique() > 1)
    ):
        stat, pval = mannwhitneyu(human_vals, random_vals, alternative='two-sided')
        if not pd.isna(pval):
            # Find the maximum value for this region across all models
            region_data = df_all_melted[df_all_melted['Region'] == region]['Attribution'].dropna()
            region_max = region_data.max()
            
            # Position p-value well above the data
            y_text = region_max * 3  # Multiply by 3 for good separation on log scale
            
            # Format p-value for better readability
            if pval < 0.001:
                p_text = f"p<0.001"
            else:
                p_text = f"p={pval:.3f}"
            
            plt.text(
                i, y_text, p_text, ha='center', va='bottom', 
                fontsize=10, color='black', fontweight='bold',
                bbox=dict(boxstyle='round,pad=0.4', facecolor='white', 
                         edgecolor='black', alpha=0.9, linewidth=1)
            )

plt.tight_layout(rect=[0, 0, 0.85, 1])

plt.savefig('all_gene_regions_comparison_models_raw.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

# Print color scheme
print("Color scheme used:")
print("Random models: Grey (#666666)")
print("Human-Borzoi models: Red (#e74c3c)")
print("Human-Decima models: Dark Red (#8b0000)")
print("Mouse-Borzoi models: Blue (#3498db)")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Set the threshold for "consistently better"
min_diff = 0.5

# Define masks for human and random models based on the 'Source' column
human_mask = genes_combined_all['Source'].str.startswith('Human_Decima')
random_mask = genes_combined_all['Source'].str.startswith('Random')

# Group by gene and get the min Pearson for human, max Pearson for random
human_pearsons = genes_combined_all[human_mask].groupby('gene')['pearson'].min()
random_pearsons = genes_combined_all[random_mask].groupby('gene')['pearson'].max()

# Only keep genes present in both
common_genes = human_pearsons.index.intersection(random_pearsons.index)
x = random_pearsons.loc[common_genes]
y = human_pearsons.loc[common_genes]

plt.figure(figsize=(8, 8))
plt.scatter(x, y, c='lightgray', label='All genes')

# Highlight genes where human is consistently better
better_human = y > x + min_diff
plt.scatter(x[better_human], y[better_human], c='red', label='Consistently better (Human)', zorder=3)

# Diagonal and threshold lines
lims = [min(x.min(), y.min()), max(x.max(), y.max())]
plt.plot(lims, lims, 'k--', label='Equal performance')
plt.plot(lims, [l + min_diff for l in lims], 'r--', alpha=0.5, label=f'Human > Random + {min_diff}')

plt.xlabel('Best Random Pearson')
plt.ylabel('Worst Human Pearson')
plt.title(f'Consistently Better Human (min_diff={min_diff})')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
better_human_list = better_human[better_human].index.tolist()
print(len(better_human_list))
better_human_list

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Set the threshold for "consistently better"
min_diff = 0.00

# Define masks for human and random models based on the 'Source' column
human_mask = genes_combined_all['Source'].str.startswith('Human_Decima')
random_mask = genes_combined_all['Source'].str.startswith('Random')

# Group by gene and get the max Pearson for human, min Pearson for random
human_pearsons = genes_combined_all[human_mask].groupby('gene')['pearson'].max()
random_pearsons = genes_combined_all[random_mask].groupby('gene')['pearson'].min()

# Only keep genes present in both
common_genes = human_pearsons.index.intersection(random_pearsons.index)
x = human_pearsons.loc[common_genes]
y = random_pearsons.loc[common_genes]

plt.figure(figsize=(8, 8))
plt.scatter(x, y, c='lightgray', label='All genes')

# Highlight genes where random is consistently better
better_random = y > x + min_diff
plt.scatter(x[better_random], y[better_random], c='blue', label='Consistently better (Random)', zorder=3)

# Diagonal and threshold lines
lims = [min(x.min(), y.min()), max(x.max(), y.max())]
plt.plot(lims, lims, 'k--', label='Equal performance')
plt.plot(lims, [l + min_diff for l in lims], 'b--', alpha=0.5, label=f'Random > Human + {min_diff}')

plt.xlabel('Best Human Pearson')
plt.ylabel('Worst Random Pearson')
plt.title(f'Consistently Better Random (min_diff={min_diff})')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
better_random_list = better_random[better_random].index.tolist()
print(len(better_random_list))
better_random_list

### Analyses of gene conservations

In [ ]:
gene_identities = pd.read_csv('/hpc/mydata/mathias.voges/Projects/research/seq2fun/step/daniodecima-applications-main/notebooks/5_specificity/zebrafish_human_orthologs.csv')
gene_identity_map_id = dict(zip(gene_identities['gene'], gene_identities['source_perc_id']))
gene_identity_map_pos = dict(zip(gene_identities['gene'], gene_identities['source_perc_pos']))

In [ ]:
genes_combined_all['id_perc'] = genes_combined_all['gene'].map(gene_identity_map_id)
genes_combined_all['pos_perc'] = genes_combined_all['gene'].map(gene_identity_map_pos)


In [ ]:
genes_combined_all.iloc[:1, 18:]



#.to_csv('genes_combined_all_with_identity.csv', index=False)

In [ ]:
def plot_performance_vs_identity(genes_combined_all, min_diff=0.5, figsize=(12, 8)):
    """Plot performance difference vs percentage identity using mean Pearson, with NaN->0"""
    
    # Use the correct identity column and replace NaN with 0
    identity_col = 'pos_perc'
    
    # Make a copy to avoid modifying original data
    data = genes_combined_all.copy()
    
    # Replace NaN values with 0 (representing no conservation)
    data[identity_col] = data[identity_col].fillna(0)
    
    print(f"Using column: {identity_col} (NaN values replaced with 0)")
    print(f"Identity score range: {data[identity_col].min():.2f} - {data[identity_col].max():.2f}")
    print(f"Number of genes with 0% identity: {(data[identity_col] == 0).sum()}")
    print(f"Number of genes with >0% identity: {(data[identity_col] > 0).sum()}")
    
    # Calculate performance differences (Human - Random) for each gene
    # Using mean human vs mean random
    human_mask = data['Source'].str.startswith('Human')
    random_mask = data['Source'].str.startswith('Random')

    print("Genes with Human results:", data[human_mask]['gene'].nunique())
    print("Genes with Random results:", data[random_mask]['gene'].nunique())
    #print("Genes with both:", len(common_genes))
    
    # Group by gene and get the MEAN Pearson for human and random
    human_pearsons = data[human_mask].groupby('gene')['pearson'].mean()  # MEAN human
    random_pearsons = data[random_mask].groupby('gene')['pearson'].mean()  # MEAN random
    
    # Find common genes
    common_genes = human_pearsons.index.intersection(random_pearsons.index)
    print(f"Common genes between human and random models: {len(common_genes)}")
    
    # Calculate performance difference (mean human - mean random)
    perf_diff = human_pearsons.loc[common_genes] - random_pearsons.loc[common_genes]
    
    # Get identity values for these genes (now no NaN values to worry about)
    identity_values = data.groupby('gene')[identity_col].first()
    
    # Only keep genes that have both performance and identity data
    valid_genes = perf_diff.index.intersection(identity_values.index)
    perf_diff_valid = perf_diff.loc[valid_genes]
    identity_values_valid = identity_values.loc[valid_genes]
    
    print(f"Genes with both performance and identity data: {len(valid_genes)}")
    
    if len(valid_genes) == 0:
        print("No genes with both performance and identity data found!")
        return
    
    # Create the plot
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=figsize)
    
    # Plot 1: Scatter plot of performance difference vs identity
    # Color points differently for 0% vs >0% identity
    zero_identity = identity_values_valid == 0
    non_zero_identity = identity_values_valid > 0
    
    if zero_identity.any():
        ax1.scatter(identity_values_valid[zero_identity], perf_diff_valid[zero_identity], 
                   alpha=0.6, s=30, color='red', label=f'No homolog (0%, n={zero_identity.sum()})')
    if non_zero_identity.any():
        ax1.scatter(identity_values_valid[non_zero_identity], perf_diff_valid[non_zero_identity], 
                   alpha=0.6, s=30, color='blue', label=f'With homolog (>0%, n={non_zero_identity.sum()})')
    
    # Add trend line
    z = np.polyfit(identity_values_valid, perf_diff_valid, 1)
    p = np.poly1d(z)
    x_trend = np.linspace(identity_values_valid.min(), identity_values_valid.max(), 100)
    ax1.plot(x_trend, p(x_trend), "gray", linestyle='--', alpha=0.8)
    
    # Calculate correlation
    try:
        if len(identity_values_valid) > 1 and identity_values_valid.nunique() > 1 and perf_diff_valid.nunique() > 1:
            corr_pearson, p_pearson = pearsonr(identity_values_valid, perf_diff_valid)
            corr_spearman, p_spearman = spearmanr(identity_values_valid, perf_diff_valid)
            
            ax1.text(0.05, 0.95, f'Pearson r = {corr_pearson:.3f} (p = {p_pearson:.3e})\n'
                                 f'Spearman ρ = {corr_spearman:.3f} (p = {p_spearman:.3e})', 
                    transform=ax1.transAxes, verticalalignment='top',
                    bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
        else:
            ax1.text(0.05, 0.95, 'Insufficient variation for correlation', 
                    transform=ax1.transAxes, verticalalignment='top',
                    bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    except Exception as e:
        print(f"Correlation error: {e}")
        ax1.text(0.05, 0.95, 'Correlation calculation failed', 
                transform=ax1.transAxes, verticalalignment='top',
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    
    ax1.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
    ax1.axhline(y=min_diff, color='red', linestyle='--', alpha=0.5, 
                label=f'Performance advantage threshold ({min_diff})')
    ax1.set_xlabel(f'Percentage Identity (% - with 0 for no homolog)')
    ax1.set_ylabel('Performance Difference (Mean Human - Mean Random)')
    ax1.set_title('Average Model Performance vs Evolutionary Conservation')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Plot 2: Binned analysis (with special handling for 0% identity)
    try:
        # Create bins: one for 0%, then regular bins for >0%
        non_zero_values = identity_values_valid[identity_values_valid > 0]
        
        if len(non_zero_values) > 0:
            # Create bins for non-zero values
            bins = [0, 0.1] + list(np.linspace(non_zero_values.min(), non_zero_values.max(), 7)[1:])
            identity_bins = pd.cut(identity_values_valid, bins=bins, include_lowest=True)
        else:
            # Only 0% values
            identity_bins = pd.cut(identity_values_valid, bins=2, include_lowest=True)
        
        bin_stats = pd.DataFrame({
            'mean_diff': perf_diff_valid.groupby(identity_bins).mean(),
            'std_diff': perf_diff_valid.groupby(identity_bins).std(),
            'count': perf_diff_valid.groupby(identity_bins).count(),
            'bin_center': identity_values_valid.groupby(identity_bins).mean()
        }).dropna()
        
        ax2.errorbar(bin_stats['bin_center'], bin_stats['mean_diff'], 
                    yerr=bin_stats['std_diff'], fmt='o-', capsize=5, capthick=2)
        
        # Add sample size annotations to bins
        for i, (x, y, n) in enumerate(zip(bin_stats['bin_center'], bin_stats['mean_diff'], bin_stats['count'])):
            ax2.annotate(f'n={n}', (x, y), xytext=(5, 5), textcoords='offset points', fontsize=8)
    
    except Exception as e:
        print(f"Binning error: {e}")
    
    ax2.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
    ax2.axhline(y=min_diff, color='red', linestyle='--', alpha=0.5)
    ax2.set_xlabel(f'Percentage Identity (%) - Binned')
    ax2.set_ylabel('Mean Performance Difference')
    ax2.set_title('Binned Analysis: Conservation vs Performance')
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Print summary statistics
    print(f"\nSummary Statistics:")
    print(f"Total genes analyzed: {len(valid_genes)}")
    print(f"Identity range: {identity_values_valid.min():.1f}% - {identity_values_valid.max():.1f}%")
    print(f"Performance difference range: {perf_diff_valid.min():.3f} - {perf_diff_valid.max():.3f}")
    
    # Compare 0% vs >0% identity genes
    zero_genes = perf_diff_valid[identity_values_valid == 0]
    nonzero_genes = perf_diff_valid[identity_values_valid > 0]
    
    print(f"\nComparison by conservation status:")
    print(f"Genes with 0% identity (no homolog): {len(zero_genes)}")
    if len(zero_genes) > 0:
        print(f"  Mean performance difference: {zero_genes.mean():.3f} ± {zero_genes.std():.3f}")
        print(f"  Genes with advantage >{min_diff}: {(zero_genes > min_diff).sum()}/{len(zero_genes)} ({100*(zero_genes > min_diff).sum()/len(zero_genes):.1f}%)")
    
    print(f"Genes with >0% identity (with homolog): {len(nonzero_genes)}")
    if len(nonzero_genes) > 0:
        print(f"  Mean performance difference: {nonzero_genes.mean():.3f} ± {nonzero_genes.std():.3f}")
        print(f"  Genes with advantage >{min_diff}: {(nonzero_genes > min_diff).sum()}/{len(nonzero_genes)} ({100*(nonzero_genes > min_diff).sum()/len(nonzero_genes):.1f}%)")
    
    # Overall statistics
    better_genes = perf_diff_valid > 0
    print(f"\nOverall:")
    print(f"Genes where human > random (mean): {better_genes.sum()}/{len(perf_diff_valid)} ({100*better_genes.sum()/len(perf_diff_valid):.1f}%)")
    
    advantage_genes = perf_diff_valid > min_diff
    print(f"Genes with substantial advantage (>{min_diff}): {advantage_genes.sum()}/{len(perf_diff_valid)} ({100*advantage_genes.sum()/len(perf_diff_valid):.1f}%)")
    
    return perf_diff_valid, identity_values_valid

# Call the function
perf_diff, identity_values = plot_performance_vs_identity(genes_combined_all, min_diff=0.5)

In [ ]:
def plot_performance_vs_identity(genes_combined_all, min_diff=0.5, min_pearson=0.2, figsize=(12, 8)):
    """Plot performance difference vs percentage identity using mean Pearson, with quality filter"""
    
    # Use the correct identity column and replace NaN with 0
    identity_col = 'id_perc'
    
    # Make a copy to avoid modifying original data
    data = genes_combined_all.copy()
    
    # Replace NaN values with 0 (representing no conservation)
    data[identity_col] = data[identity_col].fillna(0)
    
    print(f"Using column: {identity_col} (NaN values replaced with 0)")
    print(f"Filtering out genes with Pearson < {min_pearson}")
    
    # Calculate performance differences (Human - Random) for each gene
    human_mask = data['Source'].str.startswith('Human')
    random_mask = data['Source'].str.startswith('Random')
    
    # Group by gene and get the MEAN Pearson for human and random
    human_pearsons = data[human_mask].groupby('gene')['pearson'].mean()
    random_pearsons = data[random_mask].groupby('gene')['pearson'].mean()
    
    # Find common genes
    common_genes = human_pearsons.index.intersection(random_pearsons.index)
    print(f"Common genes between human and random models: {len(common_genes)}")
    
    # FILTER: Only keep genes where at least one model type performs decently
    # Option 1: Either human OR random performs well
    #good_performance_mask = (human_pearsons.loc[common_genes] >= min_pearson) | (random_pearsons.loc[common_genes] >= min_pearson)
    
    # Option 2: Both human AND random perform well (more stringent)
    good_performance_mask = (human_pearsons.loc[common_genes] >= min_pearson) & (random_pearsons.loc[common_genes] >= min_pearson)
    
    filtered_genes = common_genes[good_performance_mask]
    print(f"Genes passing performance filter (>={min_pearson}): {len(filtered_genes)}/{len(common_genes)} ({100*len(filtered_genes)/len(common_genes):.1f}%)")
    
    # Calculate performance difference (mean human - mean random) for filtered genes
    perf_diff = human_pearsons.loc[filtered_genes] - random_pearsons.loc[filtered_genes]
    
    # Get identity values for these genes
    identity_values = data.groupby('gene')[identity_col].first()
    
    # Only keep genes that have both performance and identity data
    valid_genes = perf_diff.index.intersection(identity_values.index)
    perf_diff_valid = perf_diff.loc[valid_genes]
    identity_values_valid = identity_values.loc[valid_genes]
    
    print(f"Final genes for analysis: {len(valid_genes)}")
    print(f"Identity score range: {identity_values_valid.min():.2f} - {identity_values_valid.max():.2f}")
    print(f"Number of genes with 0% identity: {(identity_values_valid == 0).sum()}")
    print(f"Number of genes with >0% identity: {(identity_values_valid > 0).sum()}")
    
    if len(valid_genes) == 0:
        print("No genes with both performance and identity data found!")
        return
    
    # Create the plot
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=figsize)
    
    # Plot 1: Scatter plot of performance difference vs identity
    # Color points differently for 0% vs >0% identity
    zero_identity = identity_values_valid == 0
    non_zero_identity = identity_values_valid > 0
    
    if zero_identity.any():
        ax1.scatter(identity_values_valid[zero_identity], perf_diff_valid[zero_identity], 
                   alpha=0.6, s=30, color='red', label=f'No homolog (0%, n={zero_identity.sum()})')
    if non_zero_identity.any():
        ax1.scatter(identity_values_valid[non_zero_identity], perf_diff_valid[non_zero_identity], 
                   alpha=0.6, s=30, color='blue', label=f'With homolog (>0%, n={non_zero_identity.sum()})')
    
    # Add trend line
    z = np.polyfit(identity_values_valid, perf_diff_valid, 1)
    p = np.poly1d(z)
    x_trend = np.linspace(identity_values_valid.min(), identity_values_valid.max(), 100)
    ax1.plot(x_trend, p(x_trend), "gray", linestyle='--', alpha=0.8)
    
    # Calculate correlation
    try:
        if len(identity_values_valid) > 1 and identity_values_valid.nunique() > 1 and perf_diff_valid.nunique() > 1:
            corr_pearson, p_pearson = pearsonr(identity_values_valid, perf_diff_valid)
            corr_spearman, p_spearman = spearmanr(identity_values_valid, perf_diff_valid)
            
            ax1.text(0.05, 0.95, f'Pearson r = {corr_pearson:.3f} (p = {p_pearson:.3e})\n'
                                 f'Spearman ρ = {corr_spearman:.3f} (p = {p_spearman:.3e})', 
                    transform=ax1.transAxes, verticalalignment='top',
                    bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
        else:
            ax1.text(0.05, 0.95, 'Insufficient variation for correlation', 
                    transform=ax1.transAxes, verticalalignment='top',
                    bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    except Exception as e:
        print(f"Correlation error: {e}")
        ax1.text(0.05, 0.95, 'Correlation calculation failed', 
                transform=ax1.transAxes, verticalalignment='top',
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    
    ax1.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
    # ax1.axhline(y=min_diff, color='red', linestyle='--', alpha=0.5, 
    #             label=f'Performance advantage threshold ({min_diff})')
    ax1.set_xlabel(f'Percentage Identity (% - with 0 for no homolog)')
    ax1.set_ylabel('Performance Difference (Mean Human - Mean Random)')
    ax1.set_title(f'Model Performance vs Conservation (Pearson ≥ {min_pearson})')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Plot 2: Binned analysis
    try:
        # Create bins: one for 0%, then regular bins for >0%
        non_zero_values = identity_values_valid[identity_values_valid > 0]
        
        if len(non_zero_values) > 0:
            # Create bins for non-zero values
            bins = [0, 0.1] + list(np.linspace(non_zero_values.min(), non_zero_values.max(), 7)[1:])
            identity_bins = pd.cut(identity_values_valid, bins=bins, include_lowest=True)
        else:
            # Only 0% values
            identity_bins = pd.cut(identity_values_valid, bins=2, include_lowest=True)
        
        bin_stats = pd.DataFrame({
            'mean_diff': perf_diff_valid.groupby(identity_bins).mean(),
            'std_diff': perf_diff_valid.groupby(identity_bins).std(),
            'count': perf_diff_valid.groupby(identity_bins).count(),
            'bin_center': identity_values_valid.groupby(identity_bins).mean()
        }).dropna()
        
        ax2.errorbar(bin_stats['bin_center'], bin_stats['mean_diff'], 
                    yerr=bin_stats['std_diff'], fmt='o-', capsize=5, capthick=2)
        
        # Add sample size annotations to bins
        for i, (x, y, n) in enumerate(zip(bin_stats['bin_center'], bin_stats['mean_diff'], bin_stats['count'])):
            ax2.annotate(f'n={n}', (x, y), xytext=(5, 5), textcoords='offset points', fontsize=8)
    
    except Exception as e:
        print(f"Binning error: {e}")
    
    ax2.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
    #ax2.axhline(y=min_diff, color='red', linestyle='--', alpha=0.5)
    ax2.set_xlabel(f'Percentage Identity (%) - Binned')
    ax2.set_ylabel('Mean Performance Difference')
    ax2.set_title(f'Binned Analysis (Pearson ≥ {min_pearson})')
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Print summary statistics
    print(f"\nSummary Statistics (genes with Pearson ≥ {min_pearson}):")
    print(f"Total genes analyzed: {len(valid_genes)}")
    print(f"Performance difference range: {perf_diff_valid.min():.3f} - {perf_diff_valid.max():.3f}")
    
    # Show baseline performance for context
    human_baseline = human_pearsons.loc[valid_genes]
    random_baseline = random_pearsons.loc[valid_genes]
    print(f"Human model baseline: {human_baseline.mean():.3f} ± {human_baseline.std():.3f}")
    print(f"Random model baseline: {random_baseline.mean():.3f} ± {random_baseline.std():.3f}")
    
    # Compare 0% vs >0% identity genes
    zero_genes = perf_diff_valid[identity_values_valid == 0]
    nonzero_genes = perf_diff_valid[identity_values_valid > 0]
    
    print(f"\nComparison by conservation status:")
    print(f"Genes with 0% identity (no homolog): {len(zero_genes)}")
    if len(zero_genes) > 0:
        print(f"  Mean performance difference: {zero_genes.mean():.3f} ± {zero_genes.std():.3f}")
        print(f"  Genes with advantage >{min_diff}: {(zero_genes > min_diff).sum()}/{len(zero_genes)} ({100*(zero_genes > min_diff).sum()/len(zero_genes):.1f}%)")
    
    print(f"Genes with >0% identity (with homolog): {len(nonzero_genes)}")
    if len(nonzero_genes) > 0:
        print(f"  Mean performance difference: {nonzero_genes.mean():.3f} ± {nonzero_genes.std():.3f}")
        print(f"  Genes with advantage >{min_diff}: {(nonzero_genes > min_diff).sum()}/{len(nonzero_genes)} ({100*(nonzero_genes > min_diff).sum()/len(nonzero_genes):.1f}%)")
    
    # Overall statistics
    better_genes = perf_diff_valid > 0
    print(f"\nOverall:")
    print(f"Genes where human > random (mean): {better_genes.sum()}/{len(perf_diff_valid)} ({100*better_genes.sum()/len(perf_diff_valid):.1f}%)")
    
    advantage_genes = perf_diff_valid > min_diff
    print(f"Genes with substantial advantage (>{min_diff}): {advantage_genes.sum()}/{len(perf_diff_valid)} ({100*advantage_genes.sum()/len(perf_diff_valid):.1f}%)")
    
    return perf_diff_valid, identity_values_valid

# Call the function with performance filter
perf_diff, identity_values = plot_performance_vs_identity(genes_combined_all, min_diff=0.0, min_pearson=-1)

In [ ]:
# with open('high_identity_genes_75.pkl', 'rb') as f:
#     high_identity_genes = pickle.load(f)
# with open('gene_identity_map.pkl', 'rb') as f:
#     gene_identity_map = pickle.load(f)

### Analysis of conservation and expression level confounding in predictive performance. Figure 6 in DanioDecima manuscript.

In [ ]:
def expression_matched_conservation_analysis(genes_combined_all, performance_col='pearson', 
                                           min_pearson=-1.0, n_expr_bins=6, figsize=(15, 10)):
    """
    Control for expression level when analyzing conservation-performance relationship
    by matching genes with similar expression across conservation bins
    """
    
    # Calculate performance differences - FIXED to use correct column names
    data = genes_combined_all[genes_combined_all[performance_col] >= min_pearson].copy()
    
    human_mask = data['Source'].str.contains('Human', na=False)
    random_mask = data['Source'].str.contains('Random', na=False)
    
    print(f"Human performance genes: {human_mask.sum()}")
    print(f"Random performance genes: {random_mask.sum()}")
    
    # Calculate mean performance for each gene
    human_perf = data[human_mask].groupby('index')[performance_col].mean()
    random_perf = data[random_mask].groupby('index')[performance_col].mean()
    
    print(f"Common genes: {len(set(human_perf.index) & set(random_perf.index))}")
    
    # Calculate performance difference (Human - Random)
    common_genes = list(set(human_perf.index) & set(random_perf.index))  # Convert to list
    perf_diff = human_perf.loc[common_genes] - random_perf.loc[common_genes]
    
    print(f"Performance differences calculated: {len(perf_diff)}")
    
    # Get gene features (using first occurrence of each gene)
    gene_features = data.drop_duplicates('index').set_index('index')[['pos_perc', 'mean_counts', 'n_tracks']]
    gene_features['pos_perc'] = gene_features['pos_perc'].fillna(0)
    
    print(f"Gene features calculated: {len(gene_features)}")
    
    # Find intersection
    common_for_analysis = list(set(perf_diff.index) & set(gene_features.index))  # Convert to list
    
    analysis_df = pd.DataFrame({
        'performance_diff': perf_diff.loc[common_for_analysis],
        'pos_perc': gene_features.loc[common_for_analysis, 'pos_perc'],
        'mean_counts': gene_features.loc[common_for_analysis, 'mean_counts'],
        'n_tracks': gene_features.loc[common_for_analysis, 'n_tracks']
    }).dropna()
    
    print(f"Final analysis dataframe: {len(analysis_df)} genes")
    
    if len(analysis_df) == 0:
        print("ERROR: No genes available for analysis!")
        return None, None
    
    # Create expression bins
    analysis_df['expr_bin'] = pd.qcut(analysis_df['mean_counts'], q=n_expr_bins, 
                                     labels=[f'Expr_Q{i+1}' for i in range(n_expr_bins)])
    
    # Create conservation bins - FIXED to handle 'No homolog' category
    analysis_df['has_homolog'] = analysis_df['pos_perc'] > 0
    
    # Initialize conservation_bin as object dtype to allow mixed categories
    analysis_df['conservation_bin'] = 'No homolog'  # Default value
    
    # For genes with homologs, create conservation bins
    if analysis_df['has_homolog'].any():
        homolog_genes = analysis_df[analysis_df['has_homolog']].copy()
        if len(homolog_genes) > 0:
            # Create bins for homolog genes only
            homolog_bins = pd.cut(homolog_genes['pos_perc'], 
                                bins=3, labels=['Low', 'Medium', 'High'])
            # Assign back to the main dataframe
            analysis_df.loc[analysis_df['has_homolog'], 'conservation_bin'] = homolog_bins
    
    # Expression-matched analysis
    fig, axes = plt.subplots(2, 3, figsize=figsize)
    axes = axes.flatten()
    
    # Plot 1: Overall relationship
    ax = axes[0]
    scatter = ax.scatter(analysis_df['pos_perc'], analysis_df['performance_diff'], 
                        c=analysis_df['mean_counts'], cmap='viridis', alpha=0.6)
    ax.set_xlabel('Percentage Identity')
    ax.set_ylabel('Performance Difference (Human - Random)')
    ax.set_title('Overall: Performance vs Conservation')
    plt.colorbar(scatter, ax=ax, label='Expression Level')
    
    # Plot 2: By conservation bins
    ax = axes[1]
    conservation_order = ['No homolog', 'Low', 'Medium', 'High']
    valid_conservation_bins = [bin for bin in conservation_order if bin in analysis_df['conservation_bin'].values]
    
    boxplot_data = [analysis_df[analysis_df['conservation_bin'] == bin]['performance_diff'].values 
                   for bin in valid_conservation_bins]
    ax.boxplot(boxplot_data, labels=valid_conservation_bins)
    ax.set_xlabel('Conservation Category')
    ax.set_ylabel('Performance Difference')
    ax.set_title('Performance by Conservation Category')
    ax.tick_params(axis='x', rotation=45)
    
    # Plot 3: By expression bins
    ax = axes[2]
    expr_bins = analysis_df['expr_bin'].cat.categories
    boxplot_data = [analysis_df[analysis_df['expr_bin'] == bin]['performance_diff'].values 
                   for bin in expr_bins]
    ax.boxplot(boxplot_data, labels=expr_bins)
    ax.set_xlabel('Expression Quintile')
    ax.set_ylabel('Performance Difference')
    ax.set_title('Performance by Expression Level')
    ax.tick_params(axis='x', rotation=45)
    
    # Plot 4: Expression vs Conservation
    ax = axes[3]
    scatter = ax.scatter(analysis_df['pos_perc'], analysis_df['mean_counts'], 
                        c=analysis_df['performance_diff'], cmap='RdBu_r', alpha=0.6)
    ax.set_xlabel('Percentage Identity')
    ax.set_ylabel('Expression Level')
    ax.set_title('Expression vs Conservation')
    plt.colorbar(scatter, ax=ax, label='Performance Difference')
    
    # Plot 5: Matched analysis across expression bins
    ax = axes[4]
    colors = plt.cm.Set3(np.linspace(0, 1, len(expr_bins)))
    
    for i, expr_bin in enumerate(expr_bins):
        bin_data = analysis_df[analysis_df['expr_bin'] == expr_bin]
        if len(bin_data) > 5:  # Only plot if enough data points
            ax.scatter(bin_data['pos_perc'], bin_data['performance_diff'], 
                      color=colors[i], label=f'{expr_bin} (n={len(bin_data)})', alpha=0.7)
    
    ax.set_xlabel('Percentage Identity')
    ax.set_ylabel('Performance Difference')
    ax.set_title('Expression-Matched Analysis')
    ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    
    # Plot 6: Summary statistics
    ax = axes[5]
    summary_stats = []
    
    for expr_bin in expr_bins:
        bin_data = analysis_df[analysis_df['expr_bin'] == expr_bin]
        if len(bin_data) > 5:
            # Calculate correlation within this expression bin
            corr = bin_data['pos_perc'].corr(bin_data['performance_diff'])
            summary_stats.append({
                'Expression_Bin': expr_bin,
                'N_genes': len(bin_data),
                'Mean_Performance_Diff': bin_data['performance_diff'].mean(),
                'Correlation': corr
            })
    
    if summary_stats:
        summary_df = pd.DataFrame(summary_stats)
        bars = ax.bar(range(len(summary_df)), summary_df['Correlation'], 
                     color=colors[:len(summary_df)])
        ax.set_xlabel('Expression Quintile')
        ax.set_ylabel('Correlation (Identity vs Performance)')
        ax.set_title('Within-Expression-Bin Correlations')
        ax.set_xticks(range(len(summary_df)))
        ax.set_xticklabels(summary_df['Expression_Bin'], rotation=45)
        
        # Add value labels on bars
        for i, bar in enumerate(bars):
            height = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2., height,
                   f'{height:.3f}', ha='center', va='bottom')
    
    plt.tight_layout()
    plt.show()
    
    # Return results
    return summary_stats, analysis_df

# Updated propensity score matching function - FIXED
def propensity_score_matching(genes_combined_all, performance_col='pearson', 
                             min_pearson=-1.0, figsize=(24, 16)):
    """
    Use propensity score matching to create balanced comparison groups
    """
    from sklearn.linear_model import LogisticRegression
    from sklearn.preprocessing import StandardScaler
    from scipy.spatial.distance import cdist
    import numpy as np
    
    # Calculate performance differences - FIXED
    data = genes_combined_all[genes_combined_all[performance_col] >= min_pearson].copy()
    
    human_mask = data['Source'].str.contains('Human', na=False)
    random_mask = data['Source'].str.contains('Random', na=False)
    
    # Use 'index' column which contains the gene names
    human_perf = data[human_mask].groupby('index')[performance_col].mean()
    random_perf = data[random_mask].groupby('index')[performance_col].mean()
    
    common_genes = human_perf.index.intersection(random_perf.index)
    perf_diff = human_perf.loc[common_genes] - random_perf.loc[common_genes]
    
    # Get gene features for matching - FIXED
    gene_features = data.groupby('index').agg({
        'pos_perc': lambda x: x.fillna(0).iloc[0],
        'mean_counts': 'first',
        'n_tracks': 'first'
    })
    
    # Find intersection and convert to list - FIXED
    common_for_analysis = list(set(perf_diff.index) & set(gene_features.index))
    
    analysis_df = pd.DataFrame({
        'performance_diff': perf_diff.loc[common_for_analysis],
        'pos_perc': gene_features.loc[common_for_analysis, 'pos_perc'],
        'mean_counts': gene_features.loc[common_for_analysis, 'mean_counts'],
        'n_tracks': gene_features.loc[common_for_analysis, 'n_tracks']
    }).dropna()
    
    if len(analysis_df) == 0:
        print("No genes available for propensity score matching")
        return None, None
    
    # Create treatment indicator (has homolog vs no homolog)
    analysis_df['has_homolog'] = (analysis_df['pos_perc'] > 0).astype(int)
    
    # Features for matching (expression and other characteristics)
    matching_features = ['mean_counts', 'n_tracks']
    X = analysis_df[matching_features]
    
    # Check if we have both treatment and control groups
    treatment_counts = analysis_df['has_homolog'].value_counts()
    print(f"Treatment group sizes: {treatment_counts}")
    
    if len(treatment_counts) < 2:
        print("Only one group available - cannot perform matching")
        return None, analysis_df
    
    # Standardize features
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    # Fit propensity score model
    propensity_model = LogisticRegression()
    propensity_model.fit(X_scaled, analysis_df['has_homolog'])
    
    # Get propensity scores
    analysis_df['propensity_score'] = propensity_model.predict_proba(X_scaled)[:, 1]
    
    # Match treated (has homolog) to control (no homolog) based on propensity scores
    treated = analysis_df[analysis_df['has_homolog'] == 1]
    control = analysis_df[analysis_df['has_homolog'] == 0]
    
    if len(treated) == 0 or len(control) == 0:
        print("Not enough genes in treatment or control group for matching")
        return None, analysis_df
    
    # Find nearest neighbors
    distances = cdist(treated[['propensity_score']], control[['propensity_score']])
    matches = []
    
    for i, treated_gene in enumerate(treated.index):
        closest_control_idx = np.argmin(distances[i])
        closest_control_gene = control.index[closest_control_idx]
        match_distance = distances[i, closest_control_idx]
        
        # Only include good matches (within 0.1 propensity score units)
        if match_distance < 0.1:
            matches.append({
                'treated_gene': treated_gene,
                'control_gene': closest_control_gene,
                'treated_performance': treated.loc[treated_gene, 'performance_diff'],
                'control_performance': control.loc[closest_control_gene, 'performance_diff'],
                'treated_expression': treated.loc[treated_gene, 'mean_counts'],
                'control_expression': control.loc[closest_control_gene, 'mean_counts'],
                'match_distance': match_distance
            })
    
    if len(matches) == 0:
        print("No good matches found - try relaxing the matching criteria")
        return None, analysis_df
    
    matches_df = pd.DataFrame(matches)
    
    # Calculate matched difference
    matches_df['performance_difference'] = (matches_df['treated_performance'] - 
                                          matches_df['control_performance'])
    
    # Visualization
    fig, axes = plt.subplots(1, 3, figsize=figsize)
    
    # Plot 1: Propensity score distribution
    axes[0].hist(treated['propensity_score'], alpha=0.7, label='Has homolog', bins=20)
    axes[0].hist(control['propensity_score'], alpha=0.7, label='No homolog', bins=20)
    axes[0].set_xlabel('Propensity Score')
    axes[0].set_ylabel('Frequency')
    axes[0].set_title('Propensity Score Distribution')
    axes[0].legend()
    
    # Plot 2: Expression balance check
    axes[1].scatter(matches_df['treated_expression'], matches_df['control_expression'], alpha=0.6)
    max_expr = max(matches_df['treated_expression'].max(), matches_df['control_expression'].max())
    axes[1].plot([0, max_expr], [0, max_expr], 'r--', alpha=0.5)
    axes[1].set_xlabel('Treated Expression')
    axes[1].set_ylabel('Control Expression')
    axes[1].set_title('Expression Balance After Matching')
    
    # Plot 3: Performance difference
    axes[2].hist(matches_df['performance_difference'], bins=15, alpha=0.7)
    axes[2].axvline(x=matches_df['performance_difference'].mean(), color='red', linestyle='--', 
                   label=f'Mean: {matches_df["performance_difference"].mean():.3f}')
    axes[2].set_xlabel('Performance Difference (Treated - Control)')
    axes[2].set_ylabel('Frequency')
    axes[2].set_title('Matched Performance Differences')
    axes[2].legend()
    
    plt.tight_layout()
    plt.show()
    
    # Statistical test
    from scipy.stats import ttest_1samp
    t_stat, p_value = ttest_1samp(matches_df['performance_difference'], 0)
    
    print(f"\nMatched Analysis Results:")
    print(f"Number of matched pairs: {len(matches_df)}")
    print(f"Mean performance difference (Has homolog - No homolog): {matches_df['performance_difference'].mean():.3f}")
    print(f"Standard error: {matches_df['performance_difference'].std() / np.sqrt(len(matches_df)):.3f}")
    print(f"T-test against zero: t = {t_stat:.3f}, p = {p_value:.3e}")
    
    # Expression balance check
    expr_diff = matches_df['treated_expression'] - matches_df['control_expression']
    print(f"Expression balance (treated - control): {expr_diff.mean():.3f} ± {expr_diff.std():.3f}")
    
    return matches_df, analysis_df

In [ ]:
# 1. Expression-matched analysis (stratified by expression bins)
expr_results, expr_df = expression_matched_conservation_analysis(genes_combined_all)

# 2. Propensity score matching (creates balanced comparison groups)
matches_df, prop_df = propensity_score_matching(genes_combined_all)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

def conservation_analysis_with_proper_ordering(genes_combined_all, performance_col='pearson', 
                                             min_pearson=-1.0, n_expr_bins=4, figsize=(20, 18),
                                             analyze_pretraining_benefit=True):
    """
    Fixed version with proper quintile ordering and clear interpretation
    
    Parameters:
    - analyze_pretraining_benefit: If True, analyzes conservation vs pretraining benefit (Human - Random)
                                  If False, analyzes conservation vs absolute performance
    """
    
    # Define model groups
    model_groups = {
        'Random Init': 'Random',
        'Mouse-Borzoi': 'Mouse',
        'Human-Borzoi': 'Human_Borzoi', 
        'Human-Decima': 'Human_Decima'
    }
    
    # Colors for consistency
    model_colors = {
        'Random Init': '#d62728',
        'Mouse-Borzoi': '#ff7f0e', 
        'Human-Borzoi': '#2ca02c',
        'Human-Decima': '#1f77b4'
    }
    
    print("="*80)
    if analyze_pretraining_benefit:
        print("CONSERVATION VS PRETRAINING BENEFIT ANALYSIS")
    else:
        print("CONSERVATION VS ABSOLUTE PERFORMANCE ANALYSIS")
    print("="*80)
    
    # Calculate pretraining benefits if requested
    if analyze_pretraining_benefit:
        print("\n🔄 Calculating pretraining benefits...")
        
        # Filter data
        data = genes_combined_all[genes_combined_all[performance_col] >= min_pearson].copy()
        
        # Calculate performance differences for each comparison
        human_borzoi_mask = data['Source'].str.contains('Human_Borzoi', na=False)
        human_decima_mask = data['Source'].str.contains('Human_Decima', na=False)
        mouse_mask = data['Source'].str.contains('Mouse', na=False)
        random_mask = data['Source'].str.contains('Random', na=False)
        
        # Calculate mean performance for each gene by model type
        human_borzoi_perf = data[human_borzoi_mask].groupby('index')[performance_col].mean()
        human_decima_perf = data[human_decima_mask].groupby('index')[performance_col].mean()
        mouse_perf = data[mouse_mask].groupby('index')[performance_col].mean()
        random_perf = data[random_mask].groupby('index')[performance_col].mean()
        
        print(f"   • Human-Borzoi genes: {len(human_borzoi_perf)}")
        print(f"   • Human-Decima genes: {len(human_decima_perf)}")
        print(f"   • Mouse-Borzoi genes: {len(mouse_perf)}")
        print(f"   • Random genes: {len(random_perf)}")
        
        # Calculate benefits (pretrained - random)
        pretraining_benefits = {}
        
        # Human-Borzoi vs Random
        common_genes = human_borzoi_perf.index.intersection(random_perf.index)
        if len(common_genes) > 0:
            pretraining_benefits['Human-Borzoi'] = human_borzoi_perf.loc[common_genes] - random_perf.loc[common_genes]
            print(f"   • Human-Borzoi vs Random: {len(common_genes)} common genes")
        
        # Human-Decima vs Random  
        common_genes = human_decima_perf.index.intersection(random_perf.index)
        if len(common_genes) > 0:
            pretraining_benefits['Human-Decima'] = human_decima_perf.loc[common_genes] - random_perf.loc[common_genes]
            print(f"   • Human-Decima vs Random: {len(common_genes)} common genes")
        
        # Mouse-Borzoi vs Random
        common_genes = mouse_perf.index.intersection(random_perf.index)
        if len(common_genes) > 0:
            pretraining_benefits['Mouse-Borzoi'] = mouse_perf.loc[common_genes] - random_perf.loc[common_genes]
            print(f"   • Mouse-Borzoi vs Random: {len(common_genes)} common genes")
        
        # Get gene features (conservation, expression, etc.)
        gene_features = data.groupby('index').agg({
            'pos_perc': lambda x: x.fillna(0).iloc[0],
            'mean_counts': 'first',
            'n_tracks': 'first'
        })
        gene_features['pos_perc'] = gene_features['pos_perc'].fillna(0)
    
    # Prepare data for each model group
    model_data = {}
    
    for model_name, source_pattern in model_groups.items():
        print(f"\n📊 Processing {model_name}...")
        
        if analyze_pretraining_benefit:
            # Use pretraining benefit data
            if model_name == 'Random Init':
                # For random init, we'll show distribution of baseline performance
                model_mask = genes_combined_all['Source'].str.contains(source_pattern, na=False)
                model_subset = genes_combined_all[model_mask & (genes_combined_all[performance_col] >= min_pearson)].copy()
                
                if len(model_subset) == 0:
                    print(f"   ⚠️  No data found for {model_name}")
                    continue
                
                gene_stats = model_subset.groupby('index').agg({
                    performance_col: 'mean',
                    'pos_perc': lambda x: x.fillna(0).iloc[0],
                    'mean_counts': 'first',
                    'n_tracks': 'first'
                }).reset_index()
                
                # For random init, performance_diff is just the baseline performance
                gene_stats['performance_diff'] = gene_stats[performance_col]
                
            else:
                # Use pretraining benefit
                if model_name not in pretraining_benefits:
                    print(f"   ⚠️  No pretraining benefit data for {model_name}")
                    continue
                
                benefit_data = pretraining_benefits[model_name]
                common_genes_with_features = benefit_data.index.intersection(gene_features.index)
                
                if len(common_genes_with_features) == 0:
                    print(f"   ⚠️  No common genes with features for {model_name}")
                    continue
                
                gene_stats = pd.DataFrame({
                    'index': common_genes_with_features,
                    'performance_diff': benefit_data.loc[common_genes_with_features],
                    'pos_perc': gene_features.loc[common_genes_with_features, 'pos_perc'],
                    'mean_counts': gene_features.loc[common_genes_with_features, 'mean_counts'],
                    'n_tracks': gene_features.loc[common_genes_with_features, 'n_tracks']
                }).reset_index(drop=True)
        else:
            # Use absolute performance (original approach)
            model_mask = genes_combined_all['Source'].str.contains(source_pattern, na=False)
            model_subset = genes_combined_all[model_mask & (genes_combined_all[performance_col] >= min_pearson)].copy()
            
            if len(model_subset) == 0:
                print(f"   ⚠️  No data found for {model_name}")
                continue
            
            # Get gene-level statistics (average across replicates)
            gene_stats = model_subset.groupby('index').agg({
                performance_col: 'mean',
                'pos_perc': lambda x: x.fillna(0).iloc[0],
                'mean_counts': 'first',
                'n_tracks': 'first'
            }).reset_index()
            
            # For absolute performance analysis, performance_diff is just the performance
            gene_stats['performance_diff'] = gene_stats[performance_col]
        
        gene_stats['pos_perc'] = gene_stats['pos_perc'].fillna(0)
        
        # Create conservation categories
        gene_stats['has_homolog'] = gene_stats['pos_perc'] > 0
        gene_stats['conservation_bin'] = 'No homolog'
        
        if gene_stats['has_homolog'].any():
            homolog_genes = gene_stats[gene_stats['has_homolog']].copy()
            if len(homolog_genes) > 10:
                try:
                    homolog_bins = pd.cut(homolog_genes['pos_perc'], 
                                        bins=3, labels=['Low', 'Medium', 'High'])
                    gene_stats.loc[gene_stats['has_homolog'], 'conservation_bin'] = homolog_bins
                except:
                    median_conservation = homolog_genes['pos_perc'].median()
                    gene_stats.loc[gene_stats['has_homolog'], 'conservation_bin'] = \
                        homolog_genes['pos_perc'].apply(lambda x: 'High' if x >= median_conservation else 'Low')
        
        # Create expression bins with PROPER ORDERING
        try:
            gene_stats['expr_bin'] = pd.qcut(gene_stats['mean_counts'], q=n_expr_bins, 
                                           labels=[f'Q{i+1}' for i in range(n_expr_bins)],
                                           duplicates='drop')
        except:
            gene_stats['expr_bin'] = pd.cut(gene_stats['mean_counts'], bins=n_expr_bins,
                                          labels=[f'Q{i+1}' for i in range(n_expr_bins)])
        
        # ENSURE PROPER CATEGORICAL ORDERING
        if hasattr(gene_stats['expr_bin'], 'cat'):
            gene_stats['expr_bin'] = gene_stats['expr_bin'].cat.reorder_categories(
                [f'Q{i+1}' for i in range(n_expr_bins)], ordered=True)
        
        model_data[model_name] = gene_stats
        
        print(f"   • Processed {len(gene_stats)} unique genes")
        print(f"   • Expression range: {gene_stats['mean_counts'].min():.1f} - {gene_stats['mean_counts'].max():.1f}")
        if analyze_pretraining_benefit and model_name != 'Random Init':
            print(f"   • Pretraining benefit range: {gene_stats['performance_diff'].min():.3f} - {gene_stats['performance_diff'].max():.3f}")
    
    if not model_data:
        print("❌ No valid model data found!")
        return None
    
    # Create visualization with proper ordering
    fig = plt.figure(figsize=figsize)
    gs = fig.add_gridspec(3, 4, 
                         hspace=0.6,
                         wspace=0.25,
                         top=0.92,
                         bottom=0.08,
                         left=0.06,
                         right=0.98)
    
    if analyze_pretraining_benefit:
        fig.suptitle('Conservation vs Pretraining Benefit Analysis by Model Group', 
                     fontsize=18, fontweight='bold', y=0.96)
        y_label = 'Pretraining Benefit'
        correlation_label = 'Conservation-Benefit\nCorrelation'
    else:
        fig.suptitle('Conservation vs Performance Analysis by Model Group', 
                     fontsize=18, fontweight='bold', y=0.96)
        y_label = 'Performance (Pearson R)'
        correlation_label = 'Conservation-Performance\nCorrelation'
    
    # Row 1: Performance/Benefit by Conservation Category
    conservation_order = ['No homolog', 'Low', 'Medium', 'High']
    
    for i, (model_name, data) in enumerate(model_data.items()):
        ax = fig.add_subplot(gs[0, i])
        
        valid_conservation_bins = [bin for bin in conservation_order if bin in data['conservation_bin'].values]
        
        if len(valid_conservation_bins) > 1:
            boxplot_data = [data[data['conservation_bin'] == bin]['performance_diff'].values 
                           for bin in valid_conservation_bins]
            
            bp = ax.boxplot(boxplot_data, labels=valid_conservation_bins, patch_artist=True)
            
            for patch in bp['boxes']:
                patch.set_facecolor(model_colors[model_name])
                patch.set_alpha(0.7)
        
        ax.set_xlabel('Conservation Category', fontweight='bold', fontsize=11)
        ax.set_ylabel(y_label, fontweight='bold', fontsize=11)
        ax.set_title(f'{model_name}', fontweight='bold', fontsize=13, pad=10)
        ax.tick_params(axis='x', rotation=45, labelsize=10)
        ax.grid(True, alpha=0.3, axis='y')
        
        # Add horizontal line at zero for pretraining benefit
        if analyze_pretraining_benefit and model_name != 'Random Init':
            ax.axhline(y=0, color='red', linestyle='--', alpha=0.5)
    
    # Row 2: Performance/Benefit by Expression Level
    for i, (model_name, data) in enumerate(model_data.items()):
        ax = fig.add_subplot(gs[1, i])
        
        # ENSURE PROPER ORDERING for expression quintiles
        expr_bins_ordered = [f'Q{i+1}' for i in range(n_expr_bins)]
        available_bins = [bin for bin in expr_bins_ordered if bin in data['expr_bin'].values]
        
        if len(available_bins) > 1:
            boxplot_data = [data[data['expr_bin'] == bin]['performance_diff'].values 
                           for bin in available_bins]
            
            bp = ax.boxplot(boxplot_data, labels=available_bins, patch_artist=True)
            
            for patch in bp['boxes']:
                patch.set_facecolor(model_colors[model_name])
                patch.set_alpha(0.7)
        
        ax.set_xlabel('Expression Quintile', fontweight='bold', fontsize=11)
        ax.set_ylabel(y_label, fontweight='bold', fontsize=11)
        ax.set_title(f'Expression Effects', fontweight='bold', fontsize=12, pad=10)
        ax.tick_params(axis='x', labelsize=10)
        ax.grid(True, alpha=0.3, axis='y')
        
        # Add horizontal line at zero for pretraining benefit
        if analyze_pretraining_benefit and model_name != 'Random Init':
            ax.axhline(y=0, color='red', linestyle='--', alpha=0.5)
    
    # Row 3: Within-Expression-Bin Correlations (PROPERLY ORDERED)
    for i, (model_name, data) in enumerate(model_data.items()):
        ax = fig.add_subplot(gs[2, i])
        
        # Calculate correlations with PROPER ORDERING
        expr_bins_ordered = [f'Q{i+1}' for i in range(n_expr_bins)]
        correlations = []
        
        for expr_bin in expr_bins_ordered:
            if expr_bin in data['expr_bin'].values:
                bin_data = data[data['expr_bin'] == expr_bin]
                if len(bin_data) > 5:
                    homolog_data = bin_data[bin_data['has_homolog']]
                    if len(homolog_data) > 5:
                        # This is the key change - now correlating with performance_diff
                        corr = homolog_data['pos_perc'].corr(homolog_data['performance_diff'])
                        if not np.isnan(corr):
                            correlations.append({
                                'expr_bin': expr_bin,
                                'correlation': corr,
                                'n_genes': len(homolog_data),
                                'expr_range': f"{bin_data['mean_counts'].min():.0f}-{bin_data['mean_counts'].max():.0f}"
                            })
        
        if correlations:
            corr_df = pd.DataFrame(correlations)
            
            # Create properly ordered bar plot
            x_positions = range(len(corr_df))
            bars = ax.bar(x_positions, corr_df['correlation'], 
                         color=model_colors[model_name], alpha=0.8)
            
            ax.set_xlabel('Expression Quintile (Low → High)', fontweight='bold', fontsize=11)
            ax.set_ylabel(correlation_label, fontweight='bold', fontsize=11)
            ax.set_title(f'Conservation Sensitivity by Expression', fontweight='bold', fontsize=12, pad=10)
            ax.set_xticks(x_positions)
            ax.set_xticklabels(corr_df['expr_bin'], fontsize=10)
            ax.grid(True, alpha=0.3, axis='y')
            ax.axhline(y=0, color='black', linestyle='-', alpha=0.5)
            
            # Adjust y-limits based on analysis type
            if analyze_pretraining_benefit:
                ax.set_ylim(-0.25, 0.35)  # Allow for negative correlations
            else:
                ax.set_ylim(-0.05, 0.35)
            
            # Add correlation values and sample sizes
            for j, (bar, row) in enumerate(zip(bars, corr_df.itertuples())):
                height = bar.get_height()
                if abs(height) > 0.02:
                    y_pos = height + 0.01 if height >= 0 else height - 0.02
                    ax.text(bar.get_x() + bar.get_width()/2., y_pos,
                           f'{height:.3f}\n(n={row.n_genes})', ha='center', 
                           va='bottom' if height >= 0 else 'top',
                           fontsize=8, fontweight='bold')
        else:
            ax.text(0.5, 0.5, 'Insufficient\nData', 
                   ha='center', va='center', transform=ax.transAxes, 
                   fontsize=12, fontweight='bold', color='gray')
            ax.set_title(f'Conservation Sensitivity by Expression', fontweight='bold', fontsize=12, pad=10)
            ax.set_xlabel('Expression Quintile (Low → High)', fontweight='bold', fontsize=11)
            ax.set_ylabel(correlation_label, fontweight='bold', fontsize=11)
    
    plt.tight_layout()
    
    # Print detailed explanation
    print("\n" + "="*80)
    print("🔍 INTERPRETATION GUIDE FOR BOTTOM ROW")
    print("="*80)
    
    print(f"\n📊 WHAT THE BOTTOM ROW SHOWS:")
    print(f"   • X-axis: Expression quintiles Q1 (lowest) → Q6 (highest)")
    if analyze_pretraining_benefit:
        print(f"   • Y-axis: Correlation between conservation % and pretraining benefit")
        print(f"   • Positive bars = conservation predicts greater benefit from pretraining")
        print(f"   • Negative bars = conservation predicts less benefit from pretraining")
        print(f"   • This shows which genes benefit most from evolutionary pretraining")
    else:
        print(f"   • Y-axis: Correlation between conservation % and performance")
        print(f"   • Higher bars = model benefits more from conservation at that expression level")
        print(f"   • This controls for expression effects on the conservation-performance relationship")
    
    print(f"\n🧬 WHY THIS MATTERS:")
    if analyze_pretraining_benefit:
        print(f"   • Tests whether evolutionary conservation predicts pretraining effectiveness")
        print(f"   • Reveals if highly conserved genes are easier/harder to improve with pretraining")
        print(f"   • Shows expression-dependent effects of evolutionary information")
    else:
        print(f"   • Tests if conservation effects are real or just due to expression confounding")
        print(f"   • Shows which expression levels benefit most from evolutionary information")
        print(f"   • Reveals if pretrained models learned conservation patterns consistently")
    
    # Calculate summary insights
    for model_name, data in model_data.items():
        print(f"\n📈 {model_name.upper()} CONSERVATION INSIGHTS:")
        
        # Overall conservation correlation
        homolog_data = data[data['has_homolog']]
        if len(homolog_data) > 10:
            overall_corr = homolog_data['pos_perc'].corr(homolog_data['performance_diff'])
            if analyze_pretraining_benefit:
                print(f"   • Overall conservation-benefit correlation: {overall_corr:.3f}")
            else:
                print(f"   • Overall conservation-performance correlation: {overall_corr:.3f}")
        
        # Expression-specific correlations
        expr_bins_ordered = [f'Q{i+1}' for i in range(n_expr_bins)]
        expr_correlations = []
        
        for expr_bin in expr_bins_ordered:
            if expr_bin in data['expr_bin'].values:
                bin_data = data[data['expr_bin'] == expr_bin]
                homolog_data = bin_data[bin_data['has_homolog']]
                if len(homolog_data) > 5:
                    corr = homolog_data['pos_perc'].corr(homolog_data['performance_diff'])
                    if not np.isnan(corr):
                        expr_correlations.append(corr)
        
        if expr_correlations:
            avg_corr = np.mean(expr_correlations)
            max_corr = max(expr_correlations)
            min_corr = min(expr_correlations)
            print(f"   • Average conservation sensitivity across expression levels: {avg_corr:.3f}")
            print(f"   • Range of conservation sensitivity: {min_corr:.3f} to {max_corr:.3f}")
            print(f"   • Conservation effect consistency: {'High' if np.std(expr_correlations) < 0.05 else 'Variable'}")
    
    return model_data, fig

# Usage examples:
# For pretraining benefit analysis (Option 2):
model_data, fig = conservation_analysis_with_proper_ordering(genes_combined_all, analyze_pretraining_benefit=True)

# For absolute performance analysis (original):
# model_data, fig = conservation_analysis_with_proper_ordering(genes_combined_all, analyze_pretraining_benefit=False)